> 在线讲义：[第 11 章 · 线代一：矩阵乘法的三种看法、秩、特征分解与正定](https://wikieden.github.io/math-and-eval-stats/lessons/11-linear-algebra-basics.html) · 本书《数学与实验统计》

# 第 11 章 · 线代一：矩阵乘法的三种看法、秩、特征分解与正定

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/11-linear-algebra-basics.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 三种看法 | 点积 / 列 / 外积 | `matmul_dot`、`matmul_columns`、`matmul_outer` | 都是 [[19, 22], [43, 50]] |
| 第二步 线性映射 | $Ae_j$ | `rotation(90) @ [1, 0]`、`np.linalg.det(A)` | [0, 1]；−2 |
| 第三步 秩 | rank、零空间 | `np.linalg.matrix_rank(M3)`、`null_vector(M3)` | 2；[1, 1, −1] |
| 第四步 特征分解 | QΛQᵀ | `np.linalg.eigh(S)`、`power_by_eig(S, 10)` | 1, 3；[[29525, 29524], …] |
| 第五步 二次型 | xᵀMx | `quadratic_form(m, [1, −1])` | 2 / −2 |

例子：A = [[1, 2], [3, 4]]、B = [[5, 6], [7, 8]]；一个秩为 2 的 3×3 矩阵；对称矩阵 S = [[2, 1], [1, 2]]。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])
M3 = np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 0.0, 1.0]])  # 第 2 行 = 2 × 第 1 行
S = np.array([[2.0, 1.0], [1.0, 2.0]])

**函数 `matmul_dot`**：看法一：C[i, j] = A 的第 i 行 · B 的第 j 列。

In [ ]:
def matmul_dot(a, b):
    """看法一：C[i, j] = A 的第 i 行 · B 的第 j 列。"""
    return np.array([[a[i] @ b[:, j] for j in range(b.shape[1])] for i in range(a.shape[0])])

**函数 `matmul_columns`**：看法二：C 的第 j 列 = A 的各列按 B[:, j] 加权求和。

In [ ]:
def matmul_columns(a, b):
    """看法二：C 的第 j 列 = A 的各列按 B[:, j] 加权求和。"""
    return np.column_stack([sum(b[k, j] * a[:, k] for k in range(a.shape[1])) for j in range(b.shape[1])])

**函数 `matmul_outer`**：看法三：C = Σ_k (A 的第 k 列) ⊗ (B 的第 k 行)，k 个秩 1 矩阵相加。

In [ ]:
def matmul_outer(a, b):
    """看法三：C = Σ_k (A 的第 k 列) ⊗ (B 的第 k 行)，k 个秩 1 矩阵相加。"""
    return sum(np.outer(a[:, k], b[k]) for k in range(a.shape[1]))

**函数 `rotation`**：平面旋转矩阵。

In [ ]:
def rotation(deg: float) -> np.ndarray:
    """平面旋转矩阵。"""
    t = np.deg2rad(deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

**函数 `null_vector`**：零空间的一个单位向量：SVD 里最小奇异值对应的右奇异向量。

In [ ]:
def null_vector(m: np.ndarray) -> np.ndarray:
    """零空间的一个单位向量：SVD 里最小奇异值对应的右奇异向量。"""
    return np.linalg.svd(m)[2][-1]

**函数 `power_by_eig`**：S^k = Q Λ^k Qᵀ（对称矩阵的特征分解）。

In [ ]:
def power_by_eig(s: np.ndarray, k: int) -> np.ndarray:
    """S^k = Q Λ^k Qᵀ（对称矩阵的特征分解）。"""
    lam, q = np.linalg.eigh(s)
    return q @ np.diag(lam**k) @ q.T

**函数 `quadratic_form`**：xᵀ M x。

In [ ]:
def quadratic_form(m: np.ndarray, x: np.ndarray) -> float:
    """xᵀ M x。"""
    return float(x @ m @ x)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 矩阵乘法的三种看法，结果一样")
for name, f in (("行·列", matmul_dot), ("列的组合", matmul_columns), ("外积之和", matmul_outer)):
    print(f"  {name}：{f(A, B).tolist()}")

In [ ]:
print("\n第二步 矩阵 = 线性映射：看它把基向量送到哪")
print(f"  旋转 90°：e1 → {np.round(rotation(90) @ [1, 0], 3).tolist()}，e2 → {np.round(rotation(90) @ [0, 1], 3).tolist()}")
print(f"  A 把 e1 送到 {(A @ [1, 0]).tolist()}（A 的第 1 列），e2 送到 {(A @ [0, 1]).tolist()}（第 2 列）；det(A) = {np.linalg.det(A):.1f}")

In [ ]:
print("\n第三步 秩：有几个方向是独立的")
v = null_vector(M3)
print(f"  M3 的秩 = {np.linalg.matrix_rank(M3)}；零空间方向 {np.round(v / v[np.argmax(abs(v))], 3).tolist()}，M3·v = {np.round(M3 @ v, 10).tolist()}")

In [ ]:
print("\n第四步 特征分解：对称矩阵 = 旋转 · 拉伸 · 转回来")
lam, q = np.linalg.eigh(S)
print(f"  S 的特征值 {lam.tolist()}，特征向量（列）{np.round(q, 4).tolist()}")
print(f"  S^10 用特征分解算 {power_by_eig(S, 10).round(1).tolist()}，直接乘 {np.linalg.matrix_power(S, 10).tolist()}")

In [ ]:
print("\n第五步 正定与二次型")
for name, m in (("S = [[2,1],[1,2]]", S), ("[[1,2],[2,1]]", np.array([[1.0, 2.0], [2.0, 1.0]]))):
    print(f"  {name}：特征值 {np.linalg.eigvalsh(m).tolist()}；x = [1, −1] 时 xᵀMx = {quadratic_form(m, np.array([1.0, -1.0])):.1f}")

## 3. 代码里才看得到的

1. **对称矩阵用 `eigh` 不用 `eig`**：`eigh` 保证特征值是实数、特征向量正交，而且按从小到大排；`eig` 对一般矩阵，可能给出复数和不排序的结果。
2. **特征向量的符号不唯一**：[−0.707, 0.707] 和 [0.707, −0.707] 都对；比较结果时要允许整体取负。
3. **秩用 SVD 判**：`matrix_rank` 数的是"大于一个小门槛的奇异值个数"，浮点误差不会让秩 2 的矩阵被误判成秩 3。

## 4. 自己改一改

1. 把 `rotation` 的角度改成 45°，算它的特征值（会是复数，为什么？）。
2. 把 M3 的第 3 行改成 [1, 0, 2]，秩变成多少？零空间还在吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 乘法三种看：点积表、列的组合、外积相加；
> 列是基向量的去处，行列式是面积的倍数；
> 秩是独立方向数，秩不满就有零空间；
> 对称矩阵转拉转，特征值全正才是朝上的碗。